# Phase B: Exploratory Data Analysis & Causal Diagnostics
**Counterfactual Business Decision & Uplift Modeling Engine**

Objective:
1. Data ingestion and schema validation.
2. Missingness and outcome rate diagnostics.
3. Pre-treatment covariate balance (Standardized Mean Differences).
4. Sample Ratio Mismatch (SRM) Chi-Square test.
5. Propensity score overlap and common support visualization.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.data.loader import UpliftDataLoader
from src.validation.validator import DataValidator
from src.causal.diagnostics import CausalDiagnostics

## 1. Load Dataset

In [2]:
loader = UpliftDataLoader(random_seed=42)
df, metadata = loader.load_data(dataset_type='synthetic', sample_size=25000)
print(f"Loaded {len(df):,} records | Features: {metadata.n_features}")
print(f"Treated: {metadata.treatment_count:,} | Control: {metadata.control_count:,}")
print(f"Naive Observed ATE: {metadata.naive_ate:+.4%}")
df.head()

Loaded 25,000 records | Features: 12
Treated: 21,253 | Control: 3,747
Naive Observed ATE: +4.5014%


,customer_id,f0,f1,f2,f3,f4,f5,f6,f7,f8,...,propensity_score,true_prob_y0,true_prob_y1,true_y0,true_y1,true_cate,segment_label,cost,margin,expected_incremental_net_value
0,cust_0000000,-0.133126,-1.564099,-0.292345,-0.746254,0.154891,-0.634081,-0.534918,0.620650,1.458784,...,0.85,0.1697,0.2894,0,0,0.11977,Persuadables,2.53,51.22,3.61
1,cust_0000001,0.809514,-0.529500,0.108478,0.226060,1.497390,0.717028,-0.785676,-0.284294,-1.055679,...,0.85,0.1315,0.3138,0,0,0.18229,Persuadables,2.69,46.70,5.82
2,cust_0000002,0.899853,-0.872591,-0.281002,-0.289648,0.943858,-0.471095,0.373476,1.431236,0.561344,...,0.85,0.2041,0.4266,1,0,0.22249,Persuadables,2.35,52.67,9.37
3,cust_0000003,0.255936,0.386857,0.441097,-1.165371,-0.838790,-0.354851,0.477866,0.364889,0.078455,...,0.85,0.0645,0.0748,0,0,0.01033,Lost Causes,2.49,51.42,-1.96
4,cust_0000004,-0.743553,0.174745,-0.479319,0.265594,-0.121662,-1.496386,0.345268,-0.632247,0.456695,...,0.85,0.0572,0.0626,0,0,0.00541,Lost Causes,2.46,54.24,-2.16


## 2. Data Validation & Leakage Check

In [3]:
validator = DataValidator()
feature_cols = [f'f{i}' for i in range(12)]
val_report = validator.validate(df, feature_cols=feature_cols)
print(f"Validation Pass: {val_report.is_valid}")
print(f"Duplicates: {val_report.duplicate_count}")
for issue in val_report.issues:
    print(f"[{issue['severity']}] {issue['check']}: {issue['message']}")

Validation Pass: True
Duplicates: 0


## 3. Covariate Balance: Standardized Mean Differences (SMD)

In [4]:
diag = CausalDiagnostics(smd_threshold=0.10)
balance = diag.check_covariate_balance(df, feature_cols)
df_balance = pd.DataFrame([b.__dict__ for b in balance])

fig_smd = px.bar(
    df_balance, x='feature', y='smd', color='is_imbalanced',
    title='Standardized Mean Differences Across Covariates',
    labels={'smd': 'SMD', 'feature': 'Feature'}
)
fig_smd.add_hline(y=0.10, line_dash='dash', line_color='red')
fig_smd.add_hline(y=-0.10, line_dash='dash', line_color='red')
fig_smd.show()

## 4. Sample Ratio Mismatch & Propensity Overlap

In [5]:
srm_res = diag.check_sample_ratio_mismatch(df, expected_treatment_ratio=0.85)
print("SRM Check:", srm_res.interpretation)

overlap_res, prop_scores = diag.check_propensity_overlap(df, feature_cols)
print("Overlap Check:", overlap_res.summary)

SRM Check: No Sample Ratio Mismatch detected (p=0.9576). Observed allocation (21,253 treated, 3,747 control) conforms to planned design.
Overlap Check: Common support bounds: [0.829, 0.873]. Extreme positivity violations: 0 (0.00%). Overlap is adequate for causal adjustment.
